In [ ]:
Here is the complete, end-to-end Python code pipeline that combines your **pre-match metadata & engineered features (Elo ratings, rolling form, head-to-head history)** with your partner's **ball-by-ball in-game features**, followed by model training, evaluation, and analysis plots.

---

### Complete Combined ML Pipeline Code

You can copy and run this full script in your notebook once your partner provides the ball-by-ball metrics aggregated at the `match_id` level.

```python

# ==========================================
# STEP 1: LOAD & BUILD YOUR PRE-MATCH FEATURES (YOUR PART)
# ==========================================
df_raw = pd.read_csv('data/cleaned_matches.csv')

# Deduplicate to match level
df_matches = df_raw.drop_duplicates(subset=['match_id']).copy()
df_matches['date'] = pd.to_datetime(df_matches['date'])
df_matches = df_matches.sort_values('date').reset_index(drop=True)

# Derive opponent relative to toss_winner
teams_per_match = df_raw.groupby('match_id')['batting_team'].unique()
def get_opponent(row):
    teams = list(teams_per_match.get(row['match_id'], []))
    tw = row['toss_winner']
    opps = [t for t in teams if t != tw]
    return opps[0] if len(opps) > 0 else None

df_matches['opponent'] = df_matches.apply(get_opponent, axis=1)
df_matches = df_matches.dropna(subset=['opponent', 'match_won_by', 'toss_winner']).reset_index(drop=True)
df_matches['target'] = (df_matches['toss_winner'] == df_matches['match_won_by']).astype(int)

# Elo & Rolling Form Engineering
all_teams = set(df_matches['toss_winner']).union(set(df_matches['opponent']))
elo_dict = {team: 1500.0 for team in all_teams}
K = 32

toss_winner_elo, opponent_elo = [], []
toss_winner_form, opponent_form = [], []
h2h_win_pcts = []
team_history = {team: [] for team in all_teams}
h2h_history = {}

for idx, row in df_matches.iterrows():
    tw = row['toss_winner']
    opp = row['opponent']
    winner = row['match_won_by']
    
    tw_elo = elo_dict.get(tw, 1500.0)
    opp_elo = elo_dict.get(opp, 1500.0)
    toss_winner_elo.append(tw_elo)
    opponent_elo.append(opp_elo)
    
    tw_hist = team_history.get(tw, [])[-5:]
    opp_hist = team_history.get(opp, [])[-5:]
    toss_winner_form.append(np.mean(tw_hist) if len(tw_hist) > 0 else 0.5)
    opponent_form.append(np.mean(opp_hist) if len(opp_hist) > 0 else 0.5)
    
    pair = tuple(sorted([str(tw), str(opp)]))
    past_h2h = h2h_history.get(pair, [])
    h2h_win_pcts.append(sum(1 for w in past_h2h if w == tw) / len(past_h2h) if past_h2h else 0.5)
        
    team_history[tw].append(1 if winner == tw else 0)
    team_history[opp].append(1 if winner == opp else 0)
    if pair not in h2h_history:
        h2h_history[pair] = []
    h2h_history[pair].append(winner)
    
    e_tw = 1 / (1 + 10 ** ((opp_elo - tw_elo) / 400))
    s_tw = 1.0 if winner == tw else 0.0
    elo_dict[tw] = tw_elo + K * (s_tw - e_tw)
    elo_dict[opp] = opponent_elo[-1] + K * ((1 - s_tw) - (1 - e_tw)) # Simplified update

df_matches['elo_diff'] = np.array(toss_winner_elo) - np.array(opponent_elo)
df_matches['toss_winner_form_5'] = toss_winner_form
df_matches['opponent_form_5'] = opponent_form
df_matches['h2h_win_pct'] = h2h_win_pcts
df_matches['is_toss_decision_field'] = (df_matches['toss_decision'] == 'field').astype(int)

# ==========================================
# STEP 2: INCORPORATE PARTNER'S BALL-BY-BALL FEATURES
# ==========================================
# Your partner extracts these features from the ball-by-ball dataset aggregated by match_id:
# - powerplay_run_rate (Runs scored in overs 1-6)
# - powerplay_wickets (Wickets lost in overs 1-6)
# - middle_overs_sr (Strike rate in middle overs)
# - dot_ball_pct (Percentage of dot balls bowled)

# Simulating partner's extracted table structure (Replace this with your partner's actual dataframe `df_ball_features`)
np.random.seed(42)
partner_features = {
    'match_id': df_matches['match_id'],
    'powerplay_run_rate': np.random.uniform(6.0, 10.5, len(df_matches)),
    'powerplay_wickets': np.random.randint(0, 4, len(df_matches)),
    'dot_ball_pct': np.random.uniform(30.0, 50.0, len(df_matches))
}
df_partner = pd.DataFrame(partner_features)

# MERGE YOUR FEATURES WITH YOUR PARTNER'S FEATURES ON `match_id`
df_combined = pd.merge(df_matches, df_partner, on='match_id', how='inner')

# ==========================================
# STEP 3: FINAL COMBINED MODEL TRAINING & EVALUATION
# ==========================================
feature_cols = [
    'elo_diff', 'toss_winner_form_5', 'opponent_form_5', 
    'h2h_win_pct', 'is_toss_decision_field',
    'powerplay_run_rate', 'powerplay_wickets', 'dot_ball_pct'
]

X = df_combined[feature_cols]
y = df_combined['target']

# Chronological split (80% train, 20% test)
split_idx = int(len(df_combined) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Train Gradient Boosting Classifier
final_model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.03, max_depth=4, random_state=42)
final_model.fit(X_train, y_train)

preds = final_model.predict(X_test)
probs = final_model.predict_proba(X_test)[:, 1]

print(f"Final Combined Model Test Accuracy: {accuracy_score(y_test, preds)*100:.2f}%")
print(f"ROC-AUC Score: {roc_auc_score(y_test, probs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds))

# ==========================================
# STEP 4: SUBSEQUENT ANALYSIS PLOTS
# ==========================================

# 1. Feature Importance Plot (Showing contribution of your features vs partner's)
plt.figure(figsize=(10, 6))
importances = final_model.feature_importances_
indices = np.argsort(importances)

plt.barh(range(len(indices)), importances[indices], color='teal', align='center')
plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
plt.xlabel('Relative Feature Importance')
plt.title('Combined Model Feature Importance (Pre-Match + Ball-by-Ball)')
plt.tight_layout()
plt.show()

# 2. Confusion Matrix Heatmap
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Final Model Confusion Matrix')
plt.show()

# 3. ROC Curve Plot
plt.figure(figsize=(6, 5))
fpr, tpr, _ = roc_curve(y_test, probs)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, probs):.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

```

---

### Summary of What This Combined Pipeline Achieves:

1. **Seamless Integration:** Merges your rigorous pre-match groundwork (`elo_diff`, historical form, head-to-head records) with your partner's in-game dynamics (`powerplay_run_rate`, `dot_ball_pct`).
2. **Advanced Metrics:** Moves beyond basic accuracy to evaluate **ROC-AUC** and probability thresholds, essential for competitive sports modeling.
3. **Visual Validation:** Automatically generates **Feature Importance**, **Confusion Matrix**, and **ROC Curve** charts to document the final iteration clearly in your notebook.

In [2]:
pip install sklearn

  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
   

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve


ModuleNotFoundError: No module named 'sklearn'